# 🏆 Capstone: The Agricultural Intelligence Brief
### *Combine everything. Make a recommendation a Cabinet Secretary can act on.*

**⏱ about 5–6 hours (spread over several days) · Uses:** *all* of Phase 5: K-Means, hierarchical clustering, DBSCAN, PCA, association rules, evaluation, profiling, communication

---

## 📨 The brief

> **From:** Dr. Wanjiku Kariuki, Head, Agricultural Transformation Unit
> **To:** You, Lead Data Scientist
> **Subject:** One integrated brief for next month's budget committee
>
> *The Cabinet Secretary is allocating next year's agricultural support budget across **four levers**: (1) farmer programmes, (2) market infrastructure, (3) commodity-specific policy, and (4) input-supply (agro-dealer) schemes. Until now, four separate teams studied these separately. **I need one brief that connects them**: who are our farmers, what do they grow, where and how do they sell, what inputs do they buy, and **where should the money go?**
> I will give you **one page** to persuade the Committee. Make the numbers count.*

## 🔗 Why this is different from Projects 1–4
The projects each answered **one** question. Real decisions need **connected evidence**:

```
   FARMERS  ──grow──▶  PRODUCTS  ──sold at──▶  MARKETS
      │                                           │
      └──────────buy────▶  INPUT BUNDLES ◀───────┘
                    (association rules)
```

The most valuable findings come from the **links**. Examples: *Which farmer segments depend on perishable crops?* *Which segments live far from markets?* *Which input bundles serve which segment?*

## 🗺️ Milestones

| # | Milestone | Data | Methods |
|---|---|---|---|
| 1 | **Farmer segments** | `farmers` | impute, log, scale, K-Means, profile |
| 2 | **Commodity groups** | `products` | log, scale, Ward |
| 3 | **Link farmers → commodities** (test a hypothesis!) | both | crosstab, groupby |
| 4 | **Market structure** | `markets` | K-Means, DBSCAN (haversine), PCA index |
| 5 | **Input bundles** | `agro_input_baskets` | Apriori, rules |
| 6 | **Integrated strategy table** | all | synthesis |
| 7 | **The one-page brief** | all | communication |

> 💡 **You may: and should: reuse your code from Projects 1–4.** The skill being graded here is **integration and judgement**, not re-typing.

## 📏 Grading rubric (100 points)

| Criterion | Pts | Excellent looks like |
|---|---:|---|
| **Technical soundness** | 25 | Correct preprocessing; *k* justified by multiple evidence; DBSCAN uses radians/haversine; rules filtered by lift |
| **Integration** | 20 | At least **two cross-dataset links**, with one **hypothesis tested** honestly (including if it fails) |
| **Insight quality** | 15 | Findings are *non-obvious*, numerically specific, and surprising where appropriate |
| **Strategy table** | 15 | Every segment has a constraint, a market channel, an input bundle and a programme: all tied to evidence |
| **The brief** | 20 | One page; plain language; numbers; clear *decision requested*; honest limitations |
| **Reproducibility** | 5 | Notebook runs top-to-bottom; assumptions stated |

In [1]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from scipy.cluster.hierarchy import linkage, fcluster
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.metrics.pairwise import haversine_distances
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules
from course_utils import set_style, load, check, profile_clusters, PALETTE
import warnings; warnings.filterwarnings("ignore")
set_style()
RANDOM_STATE = 42; EARTH_R = 6371.0

farmers, products, markets = load("farmers"), load("products"), load("markets")
agro = load("agro_input_baskets")
print({k: v.shape for k, v in dict(farmers=farmers, products=products, markets=markets, agro=agro).items()})

{'farmers': (700, 15), 'products': (48, 11), 'markets': (103, 15), 'agro': (7384, 2)}


## Milestone 1 · Farmer segments 🌽

Reuse your Project 2 pipeline. (Features: production practices only; median-impute; `log1p` farm size; scale; K-Means with a *k* you can defend.)

### 🧩 Milestone 1: Segment the farmers
**Difficulty:** ⭐⭐  |  **Time:** ~30 min

Produce:
- `f_labels`: K-Means cluster ids (array of length 700), with `f_k` clusters (3–5)
- `f_names`: dict `{cluster_id: unique segment name}`
- `fdf`: farmers (numeric NaNs filled) with columns `cluster` and `segment`
- `f_profile`: `profile_clusters(fdf, f_labels, feature_cols)` (original units)
- `f_sizes`: Series: % of farmers per segment (sums to 100)

*(Recall: exclude revenue, county, crop, ids.)*

<details><summary>💡 Hint 1</summary>

Copy your Project 2 solution: features `farm_size_acres, yield_bags_per_acre, fertilizer_kg_per_acre, irrigated, labour_days_per_acre, improved_seed, coop_member, distance_to_market_km, post_harvest_loss_pct`.

</details>
<details><summary>💡 Hint 2</summary>

Name segments from the relative profile: huge + low labour → commercial grain; irrigated + labour-intensive + high yield → horticulture; tiny + high loss + low fertiliser → subsistence; the rest → dairy/mixed co-op.

</details>
<details><summary>💡 Hint 3</summary>

`fdf['segment'] = fdf['cluster'].map(f_names)`

</details>

In [2]:
# YOUR CODE HERE

In [ ]:
# ✔ Self-check (run me after you've written your answer)
check('3 to 5 segments, one label per farmer', lambda: 3 <= f_k <= 5 and len(f_labels) == 700 and len(set(f_labels)) == f_k)
check('unique names for every cluster', lambda: set(f_names) == set(f_labels) and len(set(f_names.values())) == f_k)
check('sizes sum to 100', lambda: abs(f_sizes.sum() - 100) < 1e-6)
check('no segment is tiny (< 5%)', lambda: f_sizes.min() > 5)

## Milestone 2 · Commodity groups 🍅

Reuse Project 4: log-transform skewed positive columns, standardise, Ward linkage, cut at 5–8 groups, and **name the groups by reading their members**.

### 🧩 Milestone 2: Group the commodities
**Difficulty:** ⭐⭐  |  **Time:** ~25 min

Produce:
- `p_group`: Ward cluster ids (array of 48) with `p_k` groups
- `p_names`: dict `{group_id: name}`
- `crop_group`: a dict mapping **product name → group name** (for all 48 products)
- `perishable_group`: the **name** of the group containing `"Tomatoes"`

Sanity: Tomatoes and Kale should share a group; Maize should not be in it.

<details><summary>💡 Hint 1</summary>

`feature_cols_p = [c for c in products.columns if c != 'product']`; log10 the positive skewed ones: `avg_price_kes_per_kg`, `shelf_life_days`, `monthly_volume_tonnes` (and leave `export_share`, which contains zeros).

</details>
<details><summary>💡 Hint 2</summary>

`p_group = fcluster(linkage(Xp, 'ward'), p_k, 'maxclust')`

</details>
<details><summary>💡 Hint 3</summary>

Name each group after what you read in its member list, e.g. look for a telltale member: Maize → grains; Beans → pulses; Potatoes → roots; Tomatoes → perishables; Milk → livestock; Tea → cash crops.

</details>

In [4]:
# YOUR CODE HERE

In [ ]:
# ✔ Self-check (run me after you've written your answer)
check('5 to 8 groups over 48 products', lambda: 5 <= p_k <= 8 and len(p_group) == 48 and len(set(p_group)) == p_k)
check('every product is mapped to a group name', lambda: len(crop_group) == 48)
check('Tomatoes and Kale share a group', lambda: crop_group['Tomatoes'] == crop_group['Kale'])
check('Maize is in a different group', lambda: crop_group['Maize'] != crop_group['Tomatoes'])
check('group names are unique', lambda: len(set(p_names.values())) == p_k)

## Milestone 3 · Link farmers to commodities 🔗: **test a hypothesis**

Farmers have a `primary_crop`; products belong to groups. Join them to ask: *what do each segment's farmers grow, in terms of market behaviour?*

> 🧪 **Hypothesis from the Commodity Board:** *"Post-harvest losses are a problem of **perishable crops**. Farmers growing tomatoes, kale, etc. lose more of their harvest than other farmers. So fund cold-chain infrastructure."*
>
> **Your job is to test it with data, not to confirm it.** A good scientist reports the result even when it contradicts the boss.

### 🧩 Milestone 3: Exposure analysis and hypothesis test
**Difficulty:** ⭐⭐⭐  |  **Time:** ~30 min

1. `fdf["crop_group"] = fdf["primary_crop"].map(crop_group)`; crops not in the products table (fodder: *Napier Grass, Dairy Fodder*) → fill with `"Fodder (not traded)"`.
2. `exposure = pd.crosstab(fdf["segment"], fdf["crop_group"], normalize="index")` (each segment's crop-group mix; rows sum to 1).
3. `perishable_share` = a Series: share of each segment's farmers growing **perishable-group** crops. `most_exposed_segment` = its argmax.
4. **Test the hypothesis:** `loss_perishable` = mean `post_harvest_loss_pct` of farmers growing perishable-group crops; `loss_other` = for all others. Then `hypothesis_supported` = `loss_perishable > loss_other` (a boolean).
5. **Dig deeper:** `loss_by_segment` = mean loss per segment. Where is the *real* loss problem?

<details><summary>💡 Hint 1</summary>

`fdf['crop_group'] = fdf['primary_crop'].map(crop_group).fillna('Fodder (not traded)')`

</details>
<details><summary>💡 Hint 2</summary>

`is_per = fdf['crop_group'] == perishable_group`; `fdf.groupby('segment')[...]`: or `is_per.groupby(fdf['segment']).mean()`.

</details>
<details><summary>💡 Hint 3</summary>

If the hypothesis fails, ask *why*. Which farmers grow perishables, and what else is true of them (irrigated? co-op members? near markets?). Which farmers lose most?

</details>

In [6]:
# YOUR CODE HERE

In [ ]:
# ✔ Self-check (run me after you've written your answer)
check('exposure rows each sum to 1', lambda: np.allclose(exposure.sum(axis=1), 1))
check('the most exposed segment is horticulture', lambda: 'orticulture' in most_exposed_segment)
check('the hypothesis is NOT supported by the data', lambda: hypothesis_supported is False)
check('the highest-loss segment is the subsistence smallholders', lambda: 'ubsistence' in loss_by_segment.idxmax())

## Milestone 4 · Market structure 🏪

Reuse Project 3: **behavioural types** (K-Means on the 10 behaviour columns, k=3), **geographic regions** (DBSCAN, haversine, 40 km, min_samples 4), **isolated markets**, and a **hub per large region**.

### 🧩 Milestone 4: Market types, regions and hubs
**Difficulty:** ⭐⭐⭐  |  **Time:** ~35 min

Produce:
- `m_types`: K-Means labels (k = 3) on the scaled behaviour features
- `geo_labels`: DBSCAN labels (haversine; `eps = 40 km`; `min_samples = 4`); `n_regions`, `n_remote`
- `mk`: markets DataFrame with columns `region` and `dev_index` (PCA PC1 rescaled 0–100, oriented so higher = more developed)
- `hub_table`: one row per region with ≥ 8 markets: `region`, `n_markets`, `hub_market` (highest `dev_index`), `avg_km_to_hub`
- `remote_dist`: mean `distance_to_city_km` of the isolated markets

<details><summary>💡 Hint 1</summary>

Behaviour columns: everything numeric **except** `latitude`/`longitude`.

</details>
<details><summary>💡 Hint 2</summary>

DBSCAN: `eps=40/EARTH_R`, `metric='haversine'`, `algorithm='ball_tree'`, input `np.radians(lat/lon)`.

</details>
<details><summary>💡 Hint 3</summary>

Hub distances: `haversine_distances(radians_sub, radians_hub.reshape(1, -1)) * EARTH_R`.

</details>

In [8]:
# YOUR CODE HERE

In [ ]:
# ✔ Self-check (run me after you've written your answer)
check('3 market types over 103 markets', lambda: len(m_types) == 103 and len(set(m_types)) == 3)
check('6 to 8 regions', lambda: 6 <= n_regions <= 8)
check('5 to 12 isolated markets', lambda: 5 <= n_remote <= 12)
check('dev_index runs 0-100', lambda: abs(mk['dev_index'].min()) < 1e-9 and abs(mk['dev_index'].max() - 100) < 1e-9)
check('at least 5 hubs, all within 100 km on average', lambda: len(hub_table) >= 5 and (hub_table['avg_km_to_hub'] < 100).all())
check('isolated markets average > 100 km from a city', lambda: remote_dist > 100)

## Milestone 5 · Input bundles 🧺

Mine the agro-dealer receipts for **bundles** (itemsets of 3+ products) with strong associations.

### 🧩 Milestone 5: Find the bundles
**Difficulty:** ⭐⭐  |  **Time:** ~20 min

1. One-hot encode the baskets of `agro`.
2. `freq = apriori(..., min_support=0.03, use_colnames=True)`; `rules = association_rules(freq, metric="confidence", min_threshold=0.5)`; keep `lift >= 2`.
3. For every rule take the **full itemset** (antecedent ∪ consequent). Deduplicate and keep only itemsets with **3 or more items**. Store them as a DataFrame `bundles` with columns `items` (a sorted tuple), `lift` (the **maximum** lift over rules for that itemset) and `support`; sorted by lift descending.
4. `n_bundles` = number of such bundles.

<details><summary>💡 Hint 1</summary>

`rules['items'] = [tuple(sorted(a | c)) for a, c in zip(rules['antecedents'], rules['consequents'])]`

</details>
<details><summary>💡 Hint 2</summary>

`bundles = rules.groupby('items').agg(lift=('lift', 'max'), support=('support', 'max')).reset_index()`; then `bundles = bundles[bundles['items'].map(len) >= 3]`.

</details>
<details><summary>💡 Hint 3</summary>

`.sort_values('lift', ascending=False).reset_index(drop=True)`

</details>

In [10]:
# YOUR CODE HERE

In [ ]:
# ✔ Self-check (run me after you've written your answer)
check('at least 5 bundles of 3+ items', lambda: n_bundles >= 5)
check('every bundle has lift >= 2', lambda: (bundles['lift'] >= 2).all())
check('every bundle has support >= 0.03', lambda: (bundles['support'] >= 0.03 - 1e-9).all())
check('a maize-planting bundle exists (Hybrid Maize Seed + DAP + CAN)', lambda: any({'Hybrid Maize Seed', 'DAP Fertilizer', 'CAN Fertilizer'} <= set(i) for i in bundles['items']))

## Milestone 6 · The integrated strategy table 🧩

This is the synthesis: **one row per farmer segment**, connecting all four datasets. Every cell should be traceable to evidence in this notebook.

| Column | Source |
|---|---|
| `segment`, `share_%` | Milestone 1 |
| `key_constraint` | Profile: loss, distance, co-op, labour… |
| `product_focus` | Milestone 3 (crop-group mix) |
| `market_channel` | Milestone 4 (market type/hub the segment should sell through) |
| `input_bundle` | Milestone 5 (bundle that fits their crops) |
| `programme` | Your recommendation |

### 🧩 Milestone 6: Build the strategy table
**Difficulty:** ⭐⭐⭐  |  **Time:** ~30 min

Create a DataFrame `strategy` with **one row per farmer segment** and these columns:
`segment`, `share_%`, `key_constraint`, `product_focus`, `market_channel`, `input_bundle`, `programme`.

- `share_%` must come from `f_sizes` (numbers, rounded to 1 decimal).
- `product_focus` should name the **crop group that segment grows most** (use `exposure`): compute it, don't type it.
- All other text columns must be **non-empty** and **specific** (mention numbers or named products).

<details><summary>💡 Hint 1</summary>

`product_focus = exposure.idxmax(axis=1)` gives, for each segment, the crop group with the largest share.

</details>
<details><summary>💡 Hint 2</summary>

Constraints from profiles: subsistence → ~27% post-harvest loss, 14% co-op; horticulture → perishables, price volatility; commercial → storage & price risk; dairy/mixed → fodder & herd health.

</details>
<details><summary>💡 Hint 3</summary>

Input bundles from Milestone 5: maize planting (Hybrid Maize Seed+DAP+CAN) · crop protection (Pesticide+Sprayer+Gloves) · dairy (Dairy Meal+Mineral Lick+Acaricide) · potato (Potato Seed+DAP+Fungicide).

</details>

In [12]:
strategy = pd.DataFrame()   # build me

In [ ]:
# ✔ Self-check (run me after you've written your answer)
check('one row per segment', lambda: len(strategy) == f_k)
check('all required columns exist', lambda: {'segment', 'share_%', 'key_constraint', 'product_focus', 'market_channel', 'input_bundle', 'programme'} <= set(strategy.columns))
check('shares sum to about 100', lambda: abs(strategy['share_%'].sum() - 100) < 1)
check('no empty cells', lambda: (strategy.astype(str).apply(lambda col: col.str.len()) > 2).all().all())
check('product_focus is computed from the data (is a real crop group)', lambda: set(strategy['product_focus']) <= set(exposure.columns))

## Milestone 7 · The one-page brief ✍️

Dr. Kariuki gave you **one page**. That means ~400–500 words. Every sentence must earn its place.

### How to structure it (the "pyramid"):
1. **Answer first.** What should the Committee *do*? (1–2 sentences)
2. **Three findings**, each with a **number** and a **"so what?"**
3. **Where the money should go** (a table or 3–4 bullets)
4. **Why you can trust this** (1 short paragraph: methods in plain words, checks you did)
5. **Limits** and **next step**

### Style rules
- **No jargon**: not "DBSCAN" but "markets too isolated to be served by any hub".
- **Numbers with comparisons**: "27% vs 9–14%", not "high".
- **Include at least one result that surprised you**: and say it plainly.
- **One decision requested.**

### ✍️ The Agricultural Intelligence Brief (one page)
*Double-click this cell to edit it. Replace every `___`.*

**To:** Dr. Wanjiku Kariuki  **From:** ___  **Re:** Where next year's agricultural support budget should go

**Recommendation.** ___

**Three findings**
1. **___**: ___ (number). *So what?* ___
2. **___**: ___ (number). *So what?* ___
3. **___ (a surprise)**: ___ (number). *So what?* ___

**Where the money should go**

| Lever | Focus | Target group | Evidence |
|---|---|---|---|
| Farmer programmes | ___ | ___ | ___ |
| Market infrastructure | ___ | ___ | ___ |
| Commodity policy | ___ | ___ | ___ |
| Input-supply schemes | ___ | ___ | ___ |

**Why you can trust it.** ___

**Limits and next step.** ___

**Decision requested.** ___

## 🎤 Optional: the 5-minute talk
Present your brief to classmates: **1 slide per finding**, one slide for the strategy table, one for limits. Rehearse the *surprise*. Peers should be able to answer: *"What is the decision, and what is the evidence?"*

## 🔍 Peer-review checklist
Swap notebooks with a classmate. Tick each box (and write one sentence of feedback for any ✗):

- [ ] Features chosen sensibly; revenue/ids/labels excluded; scaling and log-transforms explained
- [ ] *k* justified with more than one kind of evidence
- [ ] DBSCAN used radians and haversine correctly
- [ ] The **hypothesis was tested honestly**
- [ ] Association rules filtered by **lift**, not just confidence
- [ ] Strategy table is specific and traceable to evidence
- [ ] The brief fits on one page and leads with the answer
- [ ] A surprising finding is stated plainly
- [ ] Limitations are real, not boilerplate

## 🚀 Beyond the course
- **Re-run it yearly**; segments drift. How would you detect when your clusters go stale?
- **Add time**: seasonal price series per commodity would enable *time-series clustering*.
- **Go supervised**: use your segments as a target and build a classifier for new farmers (Phase 6!).
- **Try HDBSCAN, Gaussian mixtures, UMAP** and see what changes.

### 🎓 Congratulations!
You can now take an unlabelled table (or a pile of receipts), find the hidden structure, test whether it is real, name it, and persuade a decision-maker to act on it. That is **unsupervised learning in practice**.